# HCL Data Science Project: Diabetic Patient Health Record Dashboard
## Phase 1 to 5: Exploratory Data Analysis & Clinical Feature Insights

**Author:** Data Science Healthcare Analytics Team  
**Dataset:** UCI Diabetes 130-US Hospitals (1999–2008) — 101,766 encounters, 47 features  
**Objective:** Predict 30-day readmission binary target (`<30` days: yes/no) and identify primary clinical risk drivers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
print('Environment initialized.')

### 1. Load Cleaned Dataset & Review High-Level Structure

In [ ]:
data_path = '../data/cleaned_diabetes_data.csv'
df = pd.read_csv(data_path, low_memory=False)
print(f'Total encounters after removing expired/hospice: {df.shape[0]:,}')
print(f'Total features: {df.shape[1]}')
df[['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'admission_type_desc', 'readmitted_binary']].head()

### 2. Missing Value Analysis (Clinical Imputation Justification)
- `weight`: >96% missing -> Dropped per rule 2.
- `payer_code` & `medical_specialty`: Heavily missing -> Categorized as 'Unknown'.
- `race`: Missing values imputed to 'Unknown'.

In [ ]:
missing_summary = pd.DataFrame({
    'Column': ['weight', 'medical_specialty', 'payer_code', 'race', 'diag_1'],
    'Missing_Rate_Pct': [96.85, 49.08, 39.56, 2.23, 0.02],
    'Imputation_Action': ['Dropped', 'Imputed to Unknown', 'Imputed to Unknown', 'Imputed to Unknown', 'Grouped to Other']
})
missing_summary

### 3. Target Distribution: 30-Day Readmission (<30 Days vs No/<30 Days)
Readmission within 30 days is an official hospital quality measure penalized under CMS Hospital Readmissions Reduction Program (HRRP).

In [ ]:
readmit_counts = df['readmitted_binary'].value_counts(normalize=True) * 100
print('30-Day Readmission Rate (%):')
print(readmit_counts)

plt.figure(figsize=(6, 4))
sns.barplot(x=['No 30-Day Readmit (0)', 'Readmitted <30 Days (1)'], y=readmit_counts.values, palette=['#3b82f6', '#ef4444'])
plt.title('30-Day Hospital Readmission Rate Class Balance')
plt.ylabel('Percentage of Encounters (%)')
plt.ylim(0, 100)
plt.show()

### 4. Readmission Rate by Primary ICD-9 Disease Family

In [ ]:
diag_readmit = df.groupby('diag_1_family')['readmitted_binary'].agg(['count', 'mean']).sort_values('mean', ascending=False)
diag_readmit['readmit_pct'] = diag_readmit['mean'] * 100
print(diag_readmit)

plt.figure(figsize=(10, 4))
sns.barplot(x=diag_readmit.index, y=diag_readmit['readmit_pct'], palette='rocket')
plt.title('30-Day Readmission Rate Across Primary Disease Families')
plt.xticks(rotation=45)
plt.ylabel('Readmission Rate (%)')
plt.tight_layout()
plt.show()

### 5. Clinical Correlation: Prior Inpatient & Emergency Visits
Prior utilization is one of the strongest predictive signals of acute readmission.

In [ ]:
df['inpatient_bucket'] = pd.cut(df['number_inpatient'], bins=[-1, 0, 1, 2, 10], labels=['0', '1', '2', '3+'])
util_readmit = df.groupby('inpatient_bucket', observed=False)['readmitted_binary'].mean() * 100

plt.figure(figsize=(7, 4))
sns.barplot(x=util_readmit.index, y=util_readmit.values, palette='viridis')
plt.title('30-Day Readmission Risk by Number of Prior Inpatient Stays')
plt.xlabel('Number of Prior Inpatient Admissions')
plt.ylabel('30-Day Readmission Rate (%)')
plt.show()

### 6. HbA1c Testing Clinical Effect
Patients who receive HbA1c testing with medication changes represent key quality intervention opportunities.

In [ ]:
hba1c_readmit = df.groupby('A1Cresult', dropna=False)['readmitted_binary'].agg(['count', 'mean'])
hba1c_readmit['readmit_pct'] = hba1c_readmit['mean'] * 100
hba1c_readmit